In [ ]:
import os, pickle
from matplotlib import pyplot as plt
plt.rcParams.update({
    "font.size": 12,
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.serif": ["Roman"],
})

if plt.rcParams["text.usetex"]:
    print("LaTeX is enabled for text rendering in matplotlib. Disable if not Latex is installed or if you encounter issues.")
else:
    print("LaTeX is disabled for text rendering in matplotlib. Enable if you have LaTeX installed and want to use it for rendering text.")

import numpy as np
from environment import BatchDistillationEnv as Environment

In [ ]:
figpath = os.path.join("data", "LVcolumn_experiment_controlled", "figs")
os.makedirs(figpath, exist_ok=True)

untrained_path = os.path.join("data", "LVcolumn_experiment_controlled", "260508_RESULTS", "untrained_agent_260508", "results", "trajectories.pkl")
trained_path = os.path.join("data", "LVcolumn_experiment_controlled", "260508_RESULTS", "trained_agent_260508", "results", "trajectories.pkl")

with open(untrained_path, "rb") as f:
    untrained_data = pickle.load(f)

with open(trained_path, "rb") as f:
    trained_data = pickle.load(f)


In [ ]:
measurement_noise = True
parametric_uncertainty = True
return_scaled_observation = False
rescale_action = False

test_seed = int(1e3)

env_settings = {
    "diverse_initial_conditions": True,
    "initial_condition_path": os.path.join("data", "LVcolumn_DAE_init", "IC_sample_1000_cleaned.pkl"),
    "parametric_uncertainty_sampling_frequency": "episode", # "transition" or "episode"
}
env_settings.update({
    "t_step": 120,
    "n_history": 0,
    "meas_noise_bool": measurement_noise,
    "parametric_uncertainty_bool": parametric_uncertainty,
    "scale_observations": return_scaled_observation,
    "rescale_actions": rescale_action,
    "t_max": 2.0 * 3600.0 
})
env_settings["diverse_initial_conditions"] = False

environment = Environment(seed = test_seed, config = env_settings)

y_data = untrained_data._x
u_data = untrained_data._u
t_data = untrained_data._time

cum_reward = 0.0
terminated = False
truncated = False
environment.reset()

u_old = u_data[0, :].reshape(-1, 1)

for idx, (y_old) in enumerate(y_data):
    if idx == y_data.shape[0] - 1:
        terminated = True
        truncated = False

    y_old = y_old.reshape(-1, 1)[:5, :]
    u_old = u_old
    y_next = y_old.reshape(-1, 1)[:5, :]
    u_next = u_old

    reward = environment._get_reward(y_old, u_old, y_next, u_next, terminated = terminated, truncated = truncated)
    cum_reward += reward

print(f"Cumulative reward of untrained agent: {cum_reward:.2f}")


x_data = trained_data._x
u_data = trained_data._u
t_data = trained_data._time

cum_reward = 0.0
terminated = False
truncated = False
environment.reset()

u_old = u_data[0, :].reshape(-1, 1)

for idx, (y_old) in enumerate(x_data):
    if idx == x_data.shape[0] - 1:
        terminated = True
        truncated = False
    
    environment.step(u_old)

    y_old = y_old.reshape(-1, 1)[:5, :]
    u_old = u_old
    y_next = y_old.reshape(-1, 1)[:5, :]
    u_next = u_old

    reward = environment._get_reward(y_old, u_old, y_next, u_next, terminated = terminated, truncated = truncated)
    cum_reward += reward

print(f"Cumulative reward of trained agent: {cum_reward:.2f}")

In [ ]:

plt.close("all")
ncols = 3
nrow = 3
alpha = 0.2
percentage_of_ylim = 0.3

ncols = 3
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 4 * scaling_factor)


fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

x_data_untrained = untrained_data._x
u_data_untrained = untrained_data._u
time_data_untrained = untrained_data._time / 60.0
t_min = time_data_untrained[0]
t_max = time_data_untrained[-1]


x_data_trained = trained_data._x
u_data_trained = trained_data._u
time_data_trained = trained_data._time / 60.0
t_min = min(t_min, time_data_trained[0])
t_max = max(t_max, time_data_trained[-1])



xlabels = ["e0_LI", "e0_WI", "e0_w_L_D_c1", "e0_w_L_B_c1", "e0_T_tr9"]
u_labels = ["e0_rr_PLS", "e0_Q_PLR_R"]


lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])
# Add the backoff
# lbx[2] += 0.005
ubx[1] = 6e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 5.0])



ax[0, 0].plot(time_data_untrained, x_data_untrained[:, 0], label = "untrained")
ax[0, 1].plot(time_data_untrained, x_data_untrained[:, 1])
ax[0, 2].plot(time_data_untrained, x_data_untrained[:, 2])
ax[1, 0].plot(time_data_untrained, x_data_untrained[:, 3])
ax[1, 1].plot(time_data_untrained, x_data_untrained[:, 4])
ax[2, 0].step(time_data_untrained, u_data_untrained[:, 0], where="post")
ax[2, 1].step(time_data_untrained, u_data_untrained[:, 1], where="post")

ax[0, 0].plot(time_data_trained, x_data_trained[:, 0], label = "trained")
ax[0, 1].plot(time_data_trained, x_data_trained[:, 1])
ax[0, 2].plot(time_data_trained, x_data_trained[:, 2])
ax[1, 0].plot(time_data_trained, x_data_trained[:, 3])
ax[1, 1].plot(time_data_trained, x_data_trained[:, 4])
ax[2, 0].step(time_data_trained, u_data_trained[:, 0], where="post")
ax[2, 1].step(time_data_trained, u_data_trained[:, 1], where="post")



ax[0, 0].set_ylabel(r"$h~/~-$")
ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[0], lbx[0] - (ubx[0] - lbx[0]), color="tab:red", alpha=alpha)
ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), ubx[0], ubx[0] + (ubx[0] - lbx[0]), color="tab:red", alpha=alpha)
ax[0, 0].set_ylim([lbx[0] - percentage_of_ylim * (ubx[0] - lbx[0]), ubx[0] + percentage_of_ylim * (ubx[0] - lbx[0])])


ax[0, 1].set_ylabel(r"$m~/~$g")
ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[1], lbx[1] - (ubx[1] - lbx[1]), color="tab:red", alpha=alpha)
ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), ubx[1], ubx[1] + (ubx[1] - lbx[1]), color="tab:red", alpha=alpha)
ax[0, 1].set_ylim([lbx[1] - percentage_of_ylim * (ubx[1] - lbx[1]), ubx[1] + percentage_of_ylim * (ubx[1] - lbx[1])])


ax[0, 2].set_ylabel(r"$w_\mathrm{P}~/~-$")
ax[0, 2].fill_between(np.array([t_min, t_max]).flatten(), lbx[2], lbx[2] - (ubx[2] - lbx[2]), color="tab:red", alpha=alpha)
ax[0, 2].fill_between(np.array([t_min, t_max]).flatten(), ubx[2], ubx[2] + (ubx[2] - lbx[2]), color="tab:red", alpha=alpha)
ax[0, 2].set_ylim([lbx[2] - percentage_of_ylim * (ubx[2] - lbx[2]), ubx[2] + percentage_of_ylim * (ubx[2] - lbx[2])])
ax[0, 2].set_yticks(np.arange(0.84, 0.901, step=0.01))


ax[1, 0].set_ylabel(r"$w_\mathrm{B}~/~-$")
ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[3], lbx[3] - (ubx[3] - lbx[3]), color="tab:red", alpha=alpha)
ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), ubx[3], ubx[3] + (ubx[3] - lbx[3]), color="tab:red", alpha=alpha)
ax[1, 0].set_ylim([lbx[3] - percentage_of_ylim * (ubx[3] - lbx[3]), ubx[3] + percentage_of_ylim * (ubx[3] - lbx[3])])


ax[1, 1].set_ylabel(r"$T_\mathrm{B}~/~$K")
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[4], lbx[4] - (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), ubx[4], ubx[4] + (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
ax[1, 1].set_ylim([lbx[4] - percentage_of_ylim * (ubx[4] - lbx[4]), ubx[4] + percentage_of_ylim * (ubx[4] - lbx[4])])

ax[2, 0].set_ylabel(r"$rr~/~-$")
ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), lbu[0], lbu[0] - (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), ubu[0], ubu[0] + (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[2, 0].set_ylim([lbu[0] - percentage_of_ylim * (ubu[0] - lbu[0]), ubu[0] + percentage_of_ylim * (ubu[0] - lbu[0])])


ax[2, 1].set_ylabel(r"$\dot{Q}~/~$kW")
ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), lbu[1], lbu[1] - (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), ubu[1], ubu[1] + (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[2, 1].set_ylim([lbu[1] - percentage_of_ylim * (ubu[1] - lbu[1]), ubu[1] + percentage_of_ylim * (ubu[1] - lbu[1])])

ax[-1, 0].set_xlabel(r"Time $t~/~$min")
ax[-1, 1].set_xlabel(r"Time $t~/~$min")
ax[-1, 2].set_xlabel(r"Time $t~/~$min")


for axis in ax.flatten():
    axis.grid(True)
    axis.set_xlim([t_min, t_max])
    axis.set_xticks(np.arange(0, t_max[0]+1, step=15))

fig.legend(loc = "outside lower center", ncol=3)


plt.suptitle(f"Experimental closed-loop trajectory with MPC")

plt.savefig(os.path.join(figpath, "closed_loop_experimental_trajectory.png"))
print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, 'closed_loop_experimental_trajectory.png')}")

In [ ]:

plt.close("all")
ncols = 2
nrow = 3
alpha = 0.2
percentage_of_ylim = 0.2

plt.close("all")
ncols = 2
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 3.5 * scaling_factor)


fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

x_data_untrained = untrained_data._x
u_data_untrained = untrained_data._u
time_data_untrained = untrained_data._time / 60.0
t_min = time_data_untrained[0]
t_max = time_data_untrained[-1]


x_data_trained = trained_data._x
u_data_trained = trained_data._u
time_data_trained = trained_data._time / 60.0
t_min = min(t_min, time_data_trained[0])
t_max = max(t_max, time_data_trained[-1])



xlabels = ["e0_LI", "e0_WI", "e0_w_L_D_c1", "e0_w_L_B_c1", "e0_T_tr9"]
u_labels = ["e0_rr_PLS", "e0_Q_PLR_R"]


lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])
ubx[1] = 6e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 5.0])




ax[0, 0].plot(time_data_untrained, x_data_untrained[:, 1] * 1e-3, label = "untrained (0)")
ax[0, 1].plot(time_data_untrained, x_data_untrained[:, 2])
ax[1, 0].plot(time_data_untrained, x_data_untrained[:, 3])
ax[1, 1].plot(time_data_untrained, x_data_untrained[:, 4] - 273.15)
ax[2, 0].step(time_data_untrained, u_data_untrained[:, 0], where="post")
ax[2, 1].step(time_data_untrained, u_data_untrained[:, 1], where="post")

ax[0, 0].plot(time_data_trained, x_data_trained[:, 1] * 1e-3, label = "trained (1091)", linestyle = 'dashed')
ax[0, 1].plot(time_data_trained, x_data_trained[:, 2], linestyle = 'dashed')
ax[1, 0].plot(time_data_trained, x_data_trained[:, 3], linestyle = 'dashed')
ax[1, 1].plot(time_data_trained, x_data_trained[:, 4] - 273.15, linestyle = 'dashed')
ax[2, 0].step(time_data_trained, u_data_trained[:, 0], where="post", linestyle = 'dashed')
ax[2, 1].step(time_data_trained, u_data_trained[:, 1], where="post", linestyle = 'dashed')


ax[0, 0].set_ylabel(r"$m_\mathrm{P}~/~$kg")
ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[1] * 1e-3, lbx[1] * 1e-3 - (ubx[1] * 1e-3 - lbx[1] * 1e-3), color="tab:red", alpha=alpha)
ax[0, 0].axhline(y = ubx[1] * 1e-3, color = "tab:red", linestyle = ":", linewidth = 2.0, label = r"$m_\mathrm{P}^\mathrm{f}$")
ax[0, 0].set_ylim([lbx[1] * 1e-3 - percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3), ubx[1] * 1e-3 + percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3)])
ax[0, 0].set_yticks(np.arange(0.0, 6.1, step=2.0))

ax[0, 1].set_ylabel(r"$w_\mathrm{P}~/~-$")
ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[2], lbx[2] - (ubx[2] - lbx[2]), color="tab:red", alpha=alpha, label = r"constraints")
ax[0, 1].set_ylim([lbx[2] - percentage_of_ylim * (ubx[2] - lbx[2]), 0.95])
ax[0, 1].set_yticks(np.arange(0.85, 0.951, step=0.02))

ax[1, 0].set_ylabel(r"$w_\mathrm{B}~/~-$")
ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[3], lbx[3] - (ubx[3] - lbx[3]), color="tab:red", alpha=alpha)
ax[1, 0].set_ylim([lbx[3] - percentage_of_ylim * (ubx[3] - lbx[3]), 0.23])
ax[1, 0].set_yticks(np.arange(0.0, 0.231, step=0.07))


ax[1, 1].set_ylabel(r"$T_\mathrm{B}~/~^{\circ}$C")
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[4] - 273.15, lbx[4] - 273.15 - (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), 100.0, 100.0 + (100.0 + 273.15 - lbx[4]), color="tab:red", alpha=alpha)
ax[1, 1].set_ylim([lbx[4] - 273.15 - percentage_of_ylim * (100.0 + 273.15 - lbx[4]), 100.0 + percentage_of_ylim * (100.0 + 273.15 - lbx[4])])




ax[2, 0].set_ylabel(r"$rr~/~-$")
ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), lbu[0], lbu[0] - (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), ubu[0], ubu[0] + (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[2, 0].set_ylim([lbu[0] - percentage_of_ylim * (ubu[0] - lbu[0]), ubu[0] + percentage_of_ylim * (ubu[0] - lbu[0])])
ax[2, 0].set_yticks(np.arange(0.7, 1.01, step=0.1))


ax[2, 1].set_ylabel(r"$\dot{Q}~/~$kW")
ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), lbu[1], lbu[1] - (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), ubu[1], ubu[1] + (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[2, 1].set_ylim([lbu[1] - percentage_of_ylim * (ubu[1] - lbu[1]), ubu[1] + percentage_of_ylim * (ubu[1] - lbu[1])])
ax[2, 1].set_yticks(np.arange(3.0, 5.1, step=0.5))

ax[-1, 0].set_xlabel(r"Time $t~/~$min")
ax[-1, 1].set_xlabel(r"Time $t~/~$min")


for axis in ax.flatten():
    axis.grid(True)
    axis.set_xlim([t_min, t_max])
    axis.set_xticks(np.arange(0, t_max[0]+1, step=15))

fig.legend(loc = "outside lower center", ncol=2)


plt.savefig(os.path.join(figpath, "closed_loop_experimental_trajectory_present.png"), dpi=1200.0)
print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, 'closed_loop_experimental_trajectory_present.png')}")
plt.savefig(os.path.join(figpath, "closed_loop_experimental_trajectory_present.pdf"), dpi=300.0)
print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, 'closed_loop_experimental_trajectory_present.pdf')}")

In [ ]:

plt.close("all")
alpha = 0.2
percentage_of_ylim = 0.2

ncols = 2
nrow = 2
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 3.5 * scaling_factor)


fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

x_data_untrained = untrained_data._x
u_data_untrained = untrained_data._u
time_data_untrained = untrained_data._time / 60.0
t_min = time_data_untrained[0]
t_max = time_data_untrained[-1]


x_data_trained = trained_data._x
u_data_trained = trained_data._u
time_data_trained = trained_data._time / 60.0
t_min = min(t_min, time_data_trained[0])
t_max = max(t_max, time_data_trained[-1])



xlabels = ["e0_LI", "e0_WI", "e0_w_L_D_c1", "e0_w_L_B_c1", "e0_T_tr9"]
u_labels = ["e0_rr_PLS", "e0_Q_PLR_R"]


lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])
# Add the backoff
# lbx[2] += 0.005
ubx[1] = 6e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 5.0])



ax[0, 0].plot(time_data_untrained, x_data_untrained[:, 1] * 1e-3, label = "Untrained (0)")
ax[0, 1].plot(time_data_untrained, x_data_untrained[:, 2])
ax[1, 0].step(time_data_untrained, u_data_untrained[:, 0], where="post")
ax[1, 1].step(time_data_untrained, u_data_untrained[:, 1], where="post")

ax[0, 0].plot(time_data_trained, x_data_trained[:, 1] * 1e-3, label = "Trained (1091)", linestyle = 'dashed')
ax[0, 1].plot(time_data_trained, x_data_trained[:, 2], linestyle = 'dashed')
ax[1, 0].step(time_data_trained, u_data_trained[:, 0], where="post", linestyle = 'dashed')
ax[1, 1].step(time_data_trained, u_data_trained[:, 1], where="post", linestyle = 'dashed')



ax[0, 0].set_ylabel(r"$m_\mathrm{P}~/~$kg")
ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[1] * 1e-3, lbx[1] * 1e-3 - (ubx[1] * 1e-3 - lbx[1] * 1e-3), color="tab:red", alpha=alpha)
ax[0, 0].axhline(y = ubx[1] * 1e-3, color = "tab:red", linestyle = ":", linewidth = 2.0, label = r"$m_{\mathrm{P},\mathrm{f}}$")
ax[0, 0].set_ylim([lbx[1] * 1e-3 - percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3), ubx[1] * 1e-3 + percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3)])
ax[0, 0].set_yticks(np.arange(0.0, 6.1, step=2.0))

ax[0, 1].set_ylabel(r"$w_\mathrm{P}~/~-$")
ax[0, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[2], lbx[2] - (ubx[2] - lbx[2]), color="tab:red", alpha=alpha, label = r"Constraints")
ax[0, 1].set_ylim([0.83, 0.95])
ax[0, 1].set_yticks(np.arange(0.83, 0.951, step=0.02))


ax[1, 0].set_ylabel(r"$rr^\mathrm{SP}~/~-$")
ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), lbu[0], lbu[0] - (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), ubu[0], ubu[0] + (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[1, 0].set_ylim([lbu[0] - percentage_of_ylim * (ubu[0] - lbu[0]), ubu[0] + percentage_of_ylim * (ubu[0] - lbu[0])])
ax[1, 0].set_yticks(np.arange(0.7, 1.01, step=0.1))


ax[1, 1].set_ylabel(r"$\dot{Q}~/~$kW")
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), lbu[1], lbu[1] - (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), ubu[1], ubu[1] + (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[1, 1].set_ylim([lbu[1] - percentage_of_ylim * (ubu[1] - lbu[1]), ubu[1] + percentage_of_ylim * (ubu[1] - lbu[1])])
ax[1, 1].set_yticks(np.arange(3.0, 5.1, step=0.5))

ax[-1, 0].set_xlabel(r"Time $t~/~$min")
ax[-1, 1].set_xlabel(r"Time $t~/~$min")

for axis in ax.flatten():
    axis.grid(True)
    axis.set_xlim([t_min, t_max])
    axis.set_xticks(np.arange(0, t_max[0]+1, step=15))

fig.legend(loc = "outside lower center", ncol=2)
plt.suptitle(r"\textbf{Real plant}")


plt.savefig(os.path.join(figpath, "closed_loop_experimental_trajectory_present_v2.png"), dpi=1200.0)
print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, 'closed_loop_experimental_trajectory_present_v2.png')}")
plt.savefig(os.path.join(figpath, "closed_loop_experimental_trajectory_present_v2.pdf"), dpi=300.0)
print(f"Saved closed-loop trajectory figure to {os.path.join(figpath, 'closed_loop_experimental_trajectory_present_v2.pdf')}")